# Why the model quenches little, and where the selection bias comes from

Follow-up A1, A2 and B4 of the tuning page ("Follow-up after the first full analysis",
2026-11-25). Inputs from `quench_diag.py` on all jobs of the physics run (phase 3 + 3b + 3e):

```
python quench_diag.py --out /data/OO-RHIC_phys -j 12 both
#   -> results/quench_diag_eloss.npz (A1, A2), results/quench_diag_bias.npz (B4)
```

**Model results, not vetted.** Centrality here is by impact parameter, with the edges from
`phys_mb` (0–20 %: b < 2.7 fm; 60–80 %: 4.7–5.6 fm). Weights σ_k/N_k per p̂_T window.

**A1, per hard outgoing parton** (`initiators/`, pstat 1000, after ISR; p_T > 10 GeV, |η| < 0.5),
in `oo_med` and `oo_vac`:
- **Energy given to the medium**: Σ E(recoils, pstat 1) − Σ E(holes, pstat −1). Every scattering is
  jet + thermal parton (the hole) → jet′ + recoil, so recoil − hole is the jet's loss. The files
  conserve energy per event (initiators = final partons + recoils − holes). It is summed in the
  parton's hemisphere (|Δφ| < π/2; this also catches ISR partons there, so it is an upper estimate)
  and in the cone (ΔR < 0.4). The shower index cannot be used to assign it, because LBT's
  outgoing shower partons carry pstat 0, which is the ISR's index space.
- **The matched jet** (the hardest R = 0.4 jet within ΔR < 0.4 of the parton), as
  r = p_T,jet / p_T,parton:
  - parton level, shower and ISR partons only (no medium response);
  - parton level with the recoils, and the holes subtracted in the cone;
  - hadron level as in `oo_analysis_rhic.py` (visible hadrons, holes and partner copies
    subtracted).
- **Jet loss** = 1 − ⟨r⟩_med / ⟨r⟩_vac at the same parton p_T. It does not depend on the
  spectrum's slope.

**A2.** Per parton and event: whether the event has a medium, MUSIC's start time, the holes in
the cone (their first and last times), the production point, and the bulk's freeze-out time at
mid-rapidity (bulk hadrons with |η_s| < 1, τ = √(t² − z²), the median per event).

**B4.** Per `oo_vac` event: the forward estimators (`n_mbd`, `n_epd` from the analysis cache) and
the string energy in the acceptances, each divided by `phys_mb` at the same N_coll. Also the
energy the hard process takes from each nucleus (`isr_sub_E_proj/_targ`), the beam remnants'
energy, and x_proj, x_targ of the leading pair.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
D = np.load("results/quench_diag_eloss.npz")
BI = np.load("results/quench_diag_bias.npz")
b20 = np.quantile(D["mb_b"], 0.2); b60, b80 = np.quantile(D["mb_b"], [0.6, 0.8])
print(f"impact-parameter classes from phys_mb: 0-20 % b < {b20:.2f} fm, 60-80 % {b60:.2f}-{b80:.2f} fm")
PT = np.array([10, 15, 20, 25, 30, 40, 55, 80.])
CLS = {"0-20 %": lambda b: b < b20, "60-80 %": lambda b: (b >= b60) & (b < b80), "0-100 %": lambda b: b >= 0}

def take(fam, bsel):
    ev = D[f"{fam}/ev"].astype(int); b = D[f"{fam}/ev_b"][ev]; m = bsel(b)
    keys = ("pt", "E", "pid", "e_med_hemi", "e_med_cone", "r_par", "r_par_resp", "r_had", "r_had_raw",
            "n_holes", "t_first", "t_last", "x0", "y0")
    out = {k: D[f"{fam}/{k}"][m] for k in keys}
    out.update(w=D[f"{fam}/ev_w"][ev][m], hm=D[f"{fam}/ev_has_medium"][ev][m], tauf=D[f"{fam}/ev_tauf"][ev][m],
               tau0=D[f"{fam}/ev_tau0"][ev][m])
    return out

def wmean(x, w):
    ok = np.isfinite(x); x, w = x[ok], w[ok]
    mu = (w * x).sum() / w.sum(); neff = w.sum() ** 2 / (w ** 2).sum()
    return mu, np.sqrt((w * (x - mu) ** 2).sum() / w.sum() / neff)

def ratio_loss(M, V, k, m, v):
    a, c = wmean(M[k][m], M["w"][m]), wmean(V["r_par" if k == "r_par_resp" else k][v], V["w"][v])
    e = 1 - a[0] / c[0]
    return e, (1 - e) * np.hypot(a[1] / a[0], c[1] / c[0])

TAB = {}
for lab, sel in CLS.items():
    M, V = take("oo_med", sel), take("oo_vac", sel)
    rows = []
    for lo, hi in zip(PT[:-1], PT[1:]):
        m = (M["pt"] >= lo) & (M["pt"] < hi); v = (V["pt"] >= lo) & (V["pt"] < hi)
        r = {"lo": lo, "hi": hi, "n": int(m.sum())}
        r["hemi"] = wmean(M["e_med_hemi"][m] / M["E"][m], M["w"][m]); r["cone"] = wmean(M["e_med_cone"][m] / M["E"][m], M["w"][m])
        for k in ("r_par", "r_par_resp", "r_had", "r_had_raw"):
            r[k] = ratio_loss(M, V, k, m, v)
        r["inter"] = wmean((M["e_med_hemi"][m] != 0).astype(float), M["w"][m])[0]
        rows.append(r)
    TAB[lab] = rows
    print(f"\n{lab} (by b): partons med {len(M['pt'])}, vac {len(V['pt'])}")
    print("  parton pT  | to medium [% of E]: hemisphere  cone | jet loss [%]: parton  parton+resp  hadron  hadron(no sub) | interacting")
    for r in rows:
        print(f"  {r['lo']:3.0f}-{r['hi']:3.0f}    |   {100*r['hemi'][0]:5.2f}±{100*r['hemi'][1]:.2f}  {100*r['cone'][0]:5.2f}±{100*r['cone'][1]:.2f} |"
              + "".join(f"  {100*r[k][0]:5.1f}±{100*r[k][1]:3.1f}" for k in ("r_par", "r_par_resp", "r_had", "r_had_raw"))
              + f"  | {r['inter']:.2f}")

## A1. How much the jets give to the medium, and how much the jet loses

Left: the energy given to the medium (recoils − holes) as a fraction of the parton's energy, in
the hemisphere and inside the cone. Middle and right: the jet loss at parton level (without and
with the medium response) and at hadron level, for 0–20 % and 60–80 %.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
xc = 0.5 * (PT[1:] + PT[:-1])
for lab, mk in (("0-20 %", "o"), ("60-80 %", "s")):
    rows = TAB[lab]
    for key, c, nm in (("hemi", "C3", "hemisphere"), ("cone", "C0", "cone R = 0.4")):
        ax[0].errorbar(xc, [100 * r[key][0] for r in rows], [100 * r[key][1] for r in rows], fmt=mk + ("-" if lab == "0-20 %" else ":"),
                       color=c, mfc="none" if lab != "0-20 %" else c, label=f"{nm}, {lab}")
for j, lab in enumerate(("0-20 %", "60-80 %")):
    rows = TAB[lab]
    for key, c, nm in (("r_par", "C1", "parton, no response"), ("r_par_resp", "C2", "parton + recoils - holes"),
                       ("r_had", "C3", "hadron (as analysed)"), ("r_had_raw", "0.5", "hadron, holes not subtracted")):
        ax[1 + j].errorbar(xc + (["r_par", "r_par_resp", "r_had", "r_had_raw"].index(key) - 1.5) * 0.6,
                           [100 * r[key][0] for r in rows], [100 * r[key][1] for r in rows], fmt="o-", ms=4, color=c, label=nm)
    ax[1 + j].axhline(0, color="k", lw=0.8); ax[1 + j].set_title(f"jet loss 1 - <r>_med/<r>_vac, {lab} (b)", fontsize=9)
    ax[1 + j].set_ylim(-3, 13)
ax[0].set_title("energy given to the medium [% of the parton's E]", fontsize=9)
for a in ax:
    a.set_xlabel("parton $p_T$ [GeV]"); a.legend(fontsize=7)
ax[0].set_ylabel("%"); ax[1].set_ylabel("%")
fig.suptitle("Figure 1. Energy loss decomposed: to the medium, out of the cone, back with the response", fontsize=11)
fig.tight_layout(); plt.show()

**Per-jet response.** The distributions of r = p_T,jet / p_T,parton (hadron level) in medium and in
vacuum, 0–20 %, and the shift of each quantile: where in the distribution the loss sits.

In [ ]:
M, V = take("oo_med", CLS["0-20 %"]), take("oo_vac", CLS["0-20 %"])
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
qs = np.linspace(0.05, 0.95, 19)
for j, (lo, hi) in enumerate(((20, 30), (30, 55))):
    m = (M["pt"] >= lo) & (M["pt"] < hi) & (M["r_had"] > 0); v = (V["pt"] >= lo) & (V["pt"] < hi) & (V["r_had"] > 0)
    bins = np.linspace(0, 1.3, 53)
    ax[j].hist(V["r_had"][v], bins, weights=V["w"][v] / V["w"][v].sum(), histtype="step", color="C0", label="oo_vac")
    ax[j].hist(M["r_had"][m], bins, weights=M["w"][m] / M["w"][m].sum(), histtype="step", color="C3", label="oo_med")
    ax[j].set_title(f"0-20 % (b), parton {lo}-{hi} GeV", fontsize=9); ax[j].set_xlabel("$p_{T,jet}/p_{T,parton}$ (hadron level)"); ax[j].legend(fontsize=8)
    dq = np.quantile(V["r_had"][v], qs) - np.quantile(M["r_had"][m], qs)
    ax[2].plot(qs, dq, "o-", label=f"parton {lo}-{hi} GeV")
    print(f"parton {lo}-{hi} GeV, quantiles 0.1/0.25/0.5/0.75/0.9 of r: med", np.round(np.quantile(M['r_had'][m], [.1, .25, .5, .75, .9]), 3),
          "vac", np.round(np.quantile(V['r_had'][v], [.1, .25, .5, .75, .9]), 3))
ax[2].axhline(0, color="k", lw=0.8); ax[2].set_xlabel("quantile of r"); ax[2].set_ylabel("r(vac) - r(med) at the quantile")
ax[2].set_title("where the loss sits (unweighted quantiles)", fontsize=9); ax[2].legend(fontsize=8)
fig.suptitle("Figure 2. The loss is in the tail: jets that already lost much in vacuum lose more", fontsize=11)
fig.tight_layout(); plt.show()

**Flavour and path length.** Gluons against quarks: the energy given to the medium and the
hadron-level jet loss, 0–20 %. And the energy given to the medium against the parton's
production radius r₀ = √(x₀² + y₀²).

In [ ]:
M, V = take("oo_med", CLS["0-20 %"]), take("oo_vac", CLS["0-20 %"])
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for lab, fm, fv, c in (("quark", np.abs(M["pid"]) < 6, np.abs(V["pid"]) < 6, "C0"), ("gluon", M["pid"] == 21, V["pid"] == 21, "C2")):
    fh, eh = [], []
    for lo, hi in zip(PT[:-1], PT[1:]):
        m = fm & (M["pt"] >= lo) & (M["pt"] < hi); v = fv & (V["pt"] >= lo) & (V["pt"] < hi)
        if m.sum() < 100 or v.sum() < 100:
            fh.append((np.nan, np.nan)); eh.append((np.nan, np.nan)); continue
        fh.append(wmean(M["e_med_hemi"][m] / M["E"][m], M["w"][m])); eh.append(ratio_loss(M, V, "r_had", m, v))
    fh, eh = np.array(fh), np.array(eh)
    ax[0].errorbar(xc, 100 * fh[:, 0], 100 * fh[:, 1], fmt="o-", color=c, label=lab)
    ax[1].errorbar(xc, 100 * eh[:, 0], 100 * eh[:, 1], fmt="o-", color=c, label=lab)
    print(f"{lab:6s} to medium [%]: " + " ".join(f"{100*a:5.1f}" for a in fh[:, 0]) + " | hadron jet loss [%]: " + " ".join(f"{100*a:5.1f}" for a in eh[:, 0]))
print("parton bins:", " ".join(f"{a:.0f}-{b:.0f}" for a, b in zip(PT[:-1], PT[1:])))
ax[0].set_title("energy to the medium, hemisphere [% of E], 0-20 %", fontsize=9); ax[1].set_title("hadron-level jet loss [%], 0-20 %", fontsize=9)
m = M["pt"] >= 20; r0 = np.hypot(M["x0"], M["y0"])[m]; fhm = (M["e_med_hemi"] / M["E"])[m]; wm_ = M["w"][m]
rb = np.array([0, 0.5, 1, 1.5, 2, 2.5, 3, 4.])
vals = [wmean(fhm[(r0 >= a) & (r0 < b)], wm_[(r0 >= a) & (r0 < b)]) if ((r0 >= a) & (r0 < b)).sum() > 100 else (np.nan, np.nan) for a, b in zip(rb[:-1], rb[1:])]
vals = np.array(vals)
ax[2].errorbar(0.5 * (rb[1:] + rb[:-1]), 100 * vals[:, 0], 100 * vals[:, 1], 0.5 * np.diff(rb), fmt="o", color="C3")
ax[2].set_xlabel("production radius $r_0$ [fm]"); ax[2].set_title("energy to the medium vs production point, 0-20 %, parton pT > 20 GeV", fontsize=9)
for a in ax[:2]:
    a.set_xlabel("parton $p_T$ [GeV]"); a.legend(fontsize=8); a.axhline(0, color="k", lw=0.8)
fig.suptitle("Figure 3. Casimir ordering at parton level, not at jet level; path-length dependence", fontsize=11)
fig.tight_layout(); plt.show()

## A2. What the jet sees

The medium's time scales in 0–20 %: MUSIC's start time per event, the times of the holes inside the
jet's cone (when the jet scatters), and the bulk's freeze-out time at mid-rapidity. The
temperature along the path and the Matter / LBT split are **not** in the files.

In [ ]:
M = take("oo_med", CLS["0-20 %"]); m = M["pt"] >= 20
print(f"0-20 % (b), parton pT > 20 GeV: {m.sum()} partons; events with a medium: {np.average(M['hm'][m], weights=M['w'][m]):.2f}; "
      f"MUSIC tau0 values: {np.unique(np.round(M['tau0'][m][np.isfinite(M['tau0'][m])], 3))} fm/c (the XML's Initial_time_tau_0 is 0.424)")
k = m & np.isfinite(M["t_last"])
print(f"partons with holes in the cone: {k.sum() / m.sum():.2f}; first hole t: median {np.median(M['t_first'][k]):.2f} fm; "
      f"last hole t: median {np.median(M['t_last'][k]):.2f}, 90 % {np.percentile(M['t_last'][k], 90):.2f} fm")
print(f"bulk freeze-out tau at |eta_s| < 1 (median per event): median {np.nanmedian(M['tauf'][m]):.2f} fm, 10-90 % {np.round(np.nanpercentile(M['tauf'][m], [10, 90]), 2)}")
fh = (M["e_med_hemi"] / M["E"])[m]
print(f"energy to the medium per parton: share with none {np.mean(fh == 0):.2f}; median {np.median(fh):.3f}, mean {fh.mean():.3f}, 95 % {np.percentile(fh, 95):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
tb = np.linspace(0, 10, 51)
ax[0].hist(M["t_first"][k], tb, histtype="step", density=True, label="first hole in the cone")
ax[0].hist(M["t_last"][k], tb, histtype="step", density=True, label="last hole in the cone")
ax[0].hist(M["tauf"][m][np.isfinite(M["tauf"][m])], tb, histtype="step", density=True, color="k", label="bulk freeze-out tau (median per event)")
ax[0].axvline(0.4, color="0.5", ls=":", label="MUSIC start 0.40 fm/c")
ax[0].set_xlabel("t, tau [fm/c]"); ax[0].legend(fontsize=8); ax[0].set_title("0-20 % (b), parton pT > 20 GeV", fontsize=9)
ax[1].hist(100 * fh, np.linspace(0, 40, 41), weights=M["w"][m], histtype="step", density=True)
ax[1].set_yscale("log"); ax[1].set_xlabel("energy to the medium, hemisphere [% of the parton's E]"); ax[1].set_title("per parton, 0-20 %, pT > 20 GeV", fontsize=9)
fig.suptitle("Figure 4. When the jets interact, and how much each one gives", fontsize=11)
fig.tight_layout(); plt.show()

## B4. The selection bias: the forward signal against p̂_T and x

In `oo_vac` jet events, each estimator is divided by its minimum-bias mean at the same N_coll
(`phys_mb`), per p̂_T window. N_coll is the same in every window (≈ 9.1). Next to it: the
energy the hard process takes from each nucleus, and the beam remnants' energy.

In [ ]:
g = lambda k: BI[f"oo_vac/{k}"]
win = g("win").astype(int); wins = sorted(set(win)); nc = g("glauber_ncoll").astype(int); mbn = BI["mb/glauber_ncoll"]
def norm(key):
    mean = np.full(nc.max() + 2, np.nan)
    for c in np.unique(nc):
        s = mbn == c
        if s.sum() >= 20:
            mean[c] = BI[f"mb/{key}"][s].mean()
    return g(key) / mean[nc]
pth = np.array([g("pthat")[win == k].mean() for k in wins])
R_ = {k: norm(k) for k in ("n_mbd", "n_epd", "strings_eloc_mbd_pos", "strings_eloc_mbd_neg", "strings_eloc_epd")}
def per_win(x):
    return np.array([np.nanmean(x[win == k]) for k in wins]), np.array([np.nanstd(x[win == k]) / np.sqrt((win == k).sum()) for k in wins])
print("mean pTHat per window: " + " ".join(f"{a:5.1f}" for a in pth))
print("N_coll mean          : " + " ".join(f"{nc[win == k].mean():5.2f}" for k in wins))
for k, v in R_.items():
    print(f"{k:22s} / mb: " + " ".join(f"{a:5.3f}" for a in per_win(v)[0]))
for k in ("isr_sub_E_proj", "isr_sub_E_targ", "remnant_E_proj", "remnant_E_targ"):
    print(f"{k:22s} GeV : " + " ".join(f"{a:5.1f}" for a in per_win(g(k))[0]))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for k, c, nm in (("n_mbd", "C3", "MBD hadrons"), ("n_epd", "C0", "EPD hadrons"), ("strings_eloc_mbd_pos", "C1", "string energy, MBD (+)"),
                 ("strings_eloc_mbd_neg", "C2", "string energy, MBD (-)"), ("strings_eloc_epd", "C4", "string energy, EPD")):
    y, e = per_win(R_[k]); ax[0].errorbar(pth, y, e, fmt="o-", color=c, label=nm)
ax[0].axhline(1, color="k", lw=0.8); ax[0].set_xlabel("mean $\\hat p_T$ of the window [GeV]"); ax[0].set_ylabel("jet events / minimum bias, same $N_{coll}$")
ax[0].set_title("forward signal in oo_vac jet events", fontsize=9); ax[0].legend(fontsize=7)
for k, c, nm in (("isr_sub_E_proj", "C3", "taken by the hard process, projectile"), ("isr_sub_E_targ", "C1", "taken, target"),
                 ("remnant_E_proj", "C0", "beam remnant, projectile"), ("remnant_E_targ", "C2", "beam remnant, target")):
    ax[1].plot(pth, per_win(g(k))[0], "o-", color=c, label=nm)
ax[1].set_xlabel("mean $\\hat p_T$ of the window [GeV]"); ax[1].set_ylabel("GeV per event"); ax[1].legend(fontsize=7)
ax[1].set_title("energy per nucleus", fontsize=9)
ptl, pts, yl, ys = g("parton_pt_lead"), g("parton_pt_sub"), g("parton_y_lead"), g("parton_y_sub")
pt = 0.5 * (ptl + pts); xs = pt / 200 * (np.exp(yl) + np.exp(ys)) + pt / 200 * (np.exp(-yl) + np.exp(-ys))
for k in wins[-5:]:
    s = win == k; qq = np.percentile(xs[s], [0, 33, 67, 100])
    xm = [np.median(xs[s & (xs >= a) & (xs <= b)]) for a, b in zip(qq[:-1], qq[1:])]
    ym = [np.nanmean(R_["n_mbd"][s & (xs >= a) & (xs <= b)]) for a, b in zip(qq[:-1], qq[1:])]
    ax[2].plot(xm, ym, "o-", label=f"window, mean pTHat {pth[wins.index(k)]:.0f} GeV")
    print(f"pTHat {pth[wins.index(k)]:4.0f} GeV: MBD / mb in terciles of x_proj + x_targ (medians {np.round(xm, 2)}): {np.round(ym, 3)}")
ax[2].set_xlabel("$x_{proj} + x_{targ}$ of the leading pair"); ax[2].set_ylabel("MBD / minimum bias"); ax[2].legend(fontsize=7)
ax[2].set_title("at fixed $\\hat p_T$: larger x, smaller signal", fontsize=9)
fig.suptitle("Figure 5. The selection bias: energy conservation through the beam remnants", fontsize=11)
fig.tight_layout(); plt.show()

## Reading (2026-11-25; model results, not vetted)

**A1. Why the quenching is small.**
- **The jets give little energy to the medium in the first place.** In 0–20 % (by b), the
  energy given to the medium (recoils − holes, hemisphere) is ≈ 10 % of the parton's energy at
  15–20 GeV, 9 % at 20–25, 5 % at 30–40 and 3.5 % at 40–55 GeV. This is an upper estimate.
  Most of it leaves the R = 0.4 cone: only ≈ 1.2–2.3 % comes back inside, and only ≈ 20 % of the
  jets have a hole in the cone.
- **The medium response is a secondary effect.** The jet loss at parton level without the response
  is 3–8 % (30–40 → 15–20 GeV). The recoils and holes in the cone return about a third of it, and
  the hadron level gives 2–5 %. Switching the response off would not reach the data.
- **The loss is in the tail.** At 30–55 GeV the 10 % quantile of p_T,jet/p_T,parton moves by
  0.04, the median by 0.02, the 90 % quantile by ≈ 0. A mean loss of 2–4 % per jet therefore
  becomes an effective, spectrum-weighted loss of ≈ 1 % (the analysis' ε). The data's effective
  4–7 % needs a mean loss several times larger.
- **Casimir ordering exists at parton level, not at jet level.** Gluons give ≈ 1.9× more energy
  to the medium than quarks (10.6 against 5.7 % at 20–30 GeV), but gluon jets lose no more at
  hadron level (3.0 against 4.5 %): their larger loss is recovered in the cone. This is why the
  jet spectra showed no flavour ordering (`jet_spectrum_slope.ipynb`).
- **Path length matters.** Partons produced within 1 fm of the centre give 2–3× more energy to the
  medium than those produced beyond 3 fm.
- **Open:** at 10–15 GeV the hadron-level loss (≈ 3 %) is well below the parton level with
  response (≈ 8 %). The hadronization (recoils and partners in iColored) adds energy back into
  low-p_T jets. Not understood yet.

**A2. What the jet sees.**
- Every 0–20 % event has a medium, and ≈ 95 % of the jets interact.
- MUSIC starts at τ = **0.40 fm/c** per event (`tau0_music`), not at the XML's
  `Initial_time_tau_0` 0.424: the earlier statement that Matter and LBT start at 0.424 needs a
  check, together with the tStart log fix.
- The scatterings in the cone happen at t ≈ 1.9–2.1 fm/c (median), 90 % before 3.4 fm/c. The bulk
  at mid-rapidity freezes out at τ ≈ 4.1 fm/c (median). The jets interact in a short, early
  window.
- The temperature along the path, and Matter's against LBT's share, are not in the files. A
  short run that keeps the hydro evolution, or logs T at the scattering points, is needed to
  compare with JEWEL's T_i = 0.26 GeV.
- Settings (job XML): α_s 0.25 in Matter and LBT, Q₀ 2 GeV, T_c 0.16 GeV, Matter's q̂
  parametrization type 1.

**B4. The selection bias: energy conservation through the beam remnants.**
- At the same N_coll (≈ 9.1 in every window), the forward signal of jet events falls steadily
  with p̂_T: MBD from 1.05 (p̂_T ≈ 6 GeV) to 0.79 (58 GeV), EPD from 1.19 to 0.86.
- The energy the hard process takes from each nucleus doubles over that range (31 → 65 GeV). The
  beam remnants' energy halves (50 → 25 GeV).
- The strings do not drive the p̂_T dependence. Their energy in the MBD and EPD acceptances is a
  constant ≈ 0.63–0.67 of minimum bias, from the string veto.
- At fixed p̂_T, larger x of the leading pair lowers the MBD signal further by ≈ 5 %. The x
  terciles of all windows fall on **one curve** in x_proj + x_targ (Figure 5, right): the forward
  signal depends on how much momentum fraction the hard process takes, whatever the p̂_T.
- Between p̂_T ≈ 25 and 46 GeV the MBD signal drops by ≈ 8 %. That is what moves the selection-only
  R_CP from 0.89 (19–36 GeV) to 0.80 (36–60 GeV). The p_T-dependent bias is confirmed as a
  property of the model.

**For the next GPU step (D9).** The limit is the energy handed to the medium, not the response.
A stronger coupling (α_s, or the q̂ scale) is the direct knob. The medium's early start and
temperature (A2, with T in hand) could matter as much in a medium this short-lived.